In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os,sys
# 强制指定你虚拟环境里的 Python 路径（关键！）
os.environ['PYSPARK_PYTHON'] = os.path.join( os.path.dirname(os.getcwd()), ".venv", "Scripts", "python.exe" )
spark=SparkSession.builder\
     .appName('SkewTableLearn')\
     .master('local[2]')\
     .config("spark.driver.host", "127.0.0.1")\
     .config("spark.driver.bindAddress", "127.0.0.1") \
     .getOrCreate()
#原始数据
data_A=[('1','a'),('1','b'),('1','c'),('2','d'),('3','e')]
data_B=[('1','北京'),('2','上海'),('3','深圳')]
A=spark.createDataFrame(data_A,['key','value'])
B=spark.createDataFrame(data_B,['key','city'])
print('----------原始大表A--------------')
A.show()
print('----------原始小表B--------------')
B.show()

----------原始大表A--------------
+---+-----+
|key|value|
+---+-----+
|  1|    a|
|  1|    b|
|  1|    c|
|  2|    d|
|  3|    e|
+---+-----+

----------原始小表B--------------
+---+----+
|key|city|
+---+----+
|  1|北京|
|  2|上海|
|  3|深圳|
+---+----+



In [2]:
#========== 1. 广播 join ==========
print('============ 广播 join 结果============' )
A.join(broadcast(B),on='key').show()


============ 广播 join 结果============
+---+-----+----+
|key|value|city|
+---+-----+----+
|  1|    a|北京|
|  1|    b|北京|
|  1|    c|北京|
|  2|    d|上海|
|  3|    e|深圳|
+---+-----+----+



In [8]:
# ========== 2. 加盐打散 ==========
A_salt=A.withColumn('salt',(rand()*2).cast('int'))\
        .withColumn('new_key',concat(col('key'),lit('_'),col('salt')))
print("===== 加盐后 A_salt =====")
A_salt.show()

===== 加盐后 A_salt =====
+---+-----+----+-------+
|key|value|salt|new_key|
+---+-----+----+-------+
|  1|    a|   0|    1_0|
|  1|    b|   0|    1_0|
|  1|    c|   1|    1_1|
|  2|    d|   1|    2_1|
|  3|    e|   1|    3_1|
+---+-----+----+-------+



In [11]:
# ========== 3. 打散后聚合 ==========
tmp = A_salt.groupBy("new_key").count()
result = tmp.withColumn("key", split(col("new_key"), "_")[0]) \
            .groupBy("key").sum("count")
print("===== 局部聚合 tmp =====")
tmp.show()
print("===== 最终聚合 result =====")
result.show()

===== 局部聚合 tmp =====
+-------+-----+
|new_key|count|
+-------+-----+
|    1_0|    2|
|    1_1|    1|
|    2_1|    1|
|    3_1|    1|
+-------+-----+

===== 最终聚合 result =====
+---+----------+
|key|sum(count)|
+---+----------+
|  3|         1|
|  1|         3|
|  2|         1|
+---+----------+



In [12]:
# ========== 4. 小表扩容 ==========
B_expand=B.crossJoin(spark.range(2).toDF('salt'))\
           .withColumn('new_key',concat(col('key'),lit('_'),col('salt')))
print("===== 小表扩容 B_expand =====")
B_expand.show()

===== 小表扩容 B_expand =====
+---+----+----+-------+
|key|city|salt|new_key|
+---+----+----+-------+
|  1|北京|   0|    1_0|
|  1|北京|   1|    1_1|
|  2|上海|   0|    2_0|
|  2|上海|   1|    2_1|
|  3|深圳|   0|    3_0|
|  3|深圳|   1|    3_1|
+---+----+----+-------+



In [13]:
# ========== 5. 扩容后 join ==========
print("===== 扩容后 join 结果 =====")
A_salt.join(B_expand,on='new_key').show()


===== 扩容后 join 结果 =====
+-------+---+-----+----+---+----+----+
|new_key|key|value|salt|key|city|salt|
+-------+---+-----+----+---+----+----+
|    1_0|  1|    a|   0|  1|北京|   0|
|    1_0|  1|    b|   0|  1|北京|   0|
|    1_1|  1|    c|   1|  1|北京|   1|
|    2_1|  2|    d|   1|  2|上海|   1|
|    3_1|  3|    e|   1|  3|深圳|   1|
+-------+---+-----+----+---+----+----+



In [14]:
# ========== 5. 扩容后 join ==========
print("===== 扩容后 join 结果 =====")
A_salt.join(B_expand,on='new_key').drop('new_key','salt').show()

===== 扩容后 join 结果 =====
+---+-----+---+----+
|key|value|key|city|
+---+-----+---+----+
|  1|    a|  1|北京|
|  1|    b|  1|北京|
|  1|    c|  1|北京|
|  2|    d|  2|上海|
|  3|    e|  3|深圳|
+---+-----+---+----+

